# Time Analysis

## Objective

Analyze claim activity and recovery performance over time to identify trends, seasonality, and operational delays.

### Business Questions

- How do claims change over time?
- Are there seasonal trends?
- Which months have the highest recovery?
- How long does the recovery process take?
- Are recoveries improving over time?

In [ ]:
#%pip install plotly --quiet

import pandas as pd
import numpy as np
import plotly.express as px

from src.preprocessing import prepare_data


In [6]:
df = prepare_data()

### Feature Engineering

In [7]:
df["Accident Year"] = df["Accident Date"].dt.year.astype("Int64")
df["Accident Month"] = df["Accident Date"].dt.month_name()
df["Accident Month No"] = df["Accident Date"].dt.month
df["Recovery Delay (Days)"] = (
    df["RCP Date"] - df["Accident Date"]
).dt.days

In [8]:
MONTH_ORDER = [
    "January","February","March","April",
    "May","June","July","August",
    "September","October","November","December"
]

In [9]:
time_summary = (
    df.groupby(["Accident Year","Accident Month","Accident Month No"])
      .agg(
          Claims=("Claim ID","count"),
          Recovery=("Recovery Amount","sum"),
          Collected=("Collected Amount","sum"),
          Remaining=("Remaining Amount","sum"),
          Avg_Delay=("Recovery Delay (Days)","mean")
      )
      .reset_index()
)

time_summary["Collection Rate"] = (
    time_summary["Collected"]
    / time_summary["Recovery"]
    *100
).round(2)

time_summary.sort_values(
    ["Accident Year","Accident Month No"],
    inplace=True
)

time_summary.head(10)

,Accident Year,Accident Month,Accident Month No,Claims,Recovery,Collected,Remaining,Avg_Delay,Collection Rate
0,1970,January,1.0,6,33246.05,1899.75,31346.3,20397.0,5.71
2,2012,June,6.0,1,16690.00,0.00,16690.0,4964.0,0.00
1,2012,December,12.0,1,10280.00,0.00,10280.0,4785.0,0.00
5,2013,January,1.0,1,75060.00,0.00,75060.0,4751.0,0.00
4,2013,February,2.0,1,15850.00,0.00,15850.0,4732.0,0.00
7,2013,March,3.0,2,25925.00,0.00,25925.0,4703.5,0.00
6,2013,July,7.0,1,11874.00,0.00,11874.0,4589.0,0.00
3,2013,December,12.0,1,9088.00,0.00,9088.0,4436.0,0.00
11,2014,February,2.0,2,19250.00,0.00,19250.0,4363.0,0.00
14,2014,March,3.0,1,21500.00,0.00,21500.0,4326.0,0.00


In [10]:
#Sorting
time_summary["Accident Month"] = pd.Categorical(
    time_summary["Accident Month"],
    categories=MONTH_ORDER,
    ordered=True
)

time_summary = time_summary.sort_values(
    ["Accident Year", "Accident Month No"]
)

In [11]:
#Convert Year
time_summary["Accident Year"] = (
    time_summary["Accident Year"]
    .astype(int)
)

In [12]:
recent_years = (
    time_summary[
        time_summary["Accident Year"] >= 2020
    ]
)

### Monthly Claims Trend

In [13]:
fig = px.line(
    recent_years,
    x="Accident Month",
    y="Claims",
    color="Accident Year",
    markers=True,
    title="Monthly Claim Trend"
)

fig.show()

### Recovery Trends

In [14]:
fig = px.line(
    recent_years,
    x="Accident Month",
    y="Recovery",
    color="Accident Year",
    markers=True,
    title="Monthly Recovery Trend"
)

fig.show()

### Collection Rate Trend

In [15]:
fig = px.line(
    recent_years,
    x="Accident Month",
    y="Collection Rate",
    color="Accident Year",
    markers=True,
    title="Collection Rate Trend"
)

fig.show()

In [16]:
heatmap = (
    df.groupby(
        [
            "Accident Year",
            "Accident Month No"
        ]
    )
    .size()
    .reset_index(name="Claims")
)

In [17]:
fig = px.histogram(
    df,
    x="Recovery Delay (Days)",
    nbins=40,
    title="Recovery Delay Distribution"
)

fig.show()

### Executive KPIs

In [18]:
print(f"""
📊 TIME ANALYSIS

Average Recovery Delay :
{df['Recovery Delay (Days)'].mean():.1f} Days

Median Recovery Delay :
{df['Recovery Delay (Days)'].median():.1f} Days

Maximum Recovery Delay :
{df['Recovery Delay (Days)'].max():.0f} Days

Minimum Recovery Delay :
{df['Recovery Delay (Days)'].min():.0f} Days
""")


📊 TIME ANALYSIS

Average Recovery Delay :
1395.9 Days

Median Recovery Delay :
446.0 Days

Maximum Recovery Delay :
20397 Days

Minimum Recovery Delay :
-672 Days



In [19]:
slow_cases = df[
    df["Recovery Delay (Days)"] > 180
]

slow_cases[
    [
        "Claim Number",
        "Officer",
        "Recovery Amount",
        "Recovery Delay (Days)"
    ]
].sort_values(
    "Recovery Delay (Days)",
    ascending=False
)

,Claim Number,Officer,Recovery Amount,Recovery Delay (Days)
2587,C/ERO1/2025/CMC/0000765,NaN,14736.5,20397.0
2586,C/ERO1/2025/CMC/0000751,NaN,1057.5,20397.0
2588,C/ERO1/2025/CMC/0000977,Munirah,1672.5,20397.0
2589,C/ERO1/2025/CMC/0001117,NaN,7249.3,20397.0
2592,C/WRO1/2025/CMC/0000310,NaN,6630.5,20397.0
...,...,...,...,...
40,C/CRO1/2026/PTRA/0002930,Jasim,2311.0,186.0
61,C/CRO1/2026/PTO/0001004,Munirah,3558.0,184.0
801,C/CRO1/2026/PTRA/0001098,Munirah,3500.0,183.0
206,C/WRO1/2026/CMC/0000313,Jasim,3000.0,182.0


### Recovery Funnel

In [20]:
total_claims = len(df)

registered = df["Claim ID"].nunique()

active = df["Status"].notna().sum()

collected = (df["Collected Amount"] > 0).sum()

fully_collected = (
    df["Remaining Amount"] <= 0
).sum()

funnel = pd.DataFrame({
    "Stage":[
        "Total Claims",
        "Processed",
        "Recovered",
        "Fully Recovered"
    ],
    "Count":[
        registered,
        active,
        collected,
        fully_collected
    ]
})

import plotly.express as px

fig = px.funnel(
    funnel,
    x="Count",
    y="Stage",
    title="Claim Recovery Funnel"
)

fig.show()